# VeloceReduction — one observing night

**Data flow**

`DetectorFrame → OrderGeometry → OrderMatrix → ExtractionResult`

For `extraction_mode="fibre"`, `FibreGeometry` is an additional Flat-derived input to the final extraction.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from astropy.io import fits
from astropy.table import Table, vstack

from velocereduction import __version__, ReductionConfig
from velocereduction import (
    observations, detector, orders, fibres, flat, extraction,
    calibration, thorium, simlc, wavelength, diagnostics, constants
)
from velocereduction.config import prepare_reduction, setup_logging

night = "001122"
# night = "260703"
config = ReductionConfig(
    night=night,
    extraction_mode="fibre",   # "summed" or "fibre"
    diagnostics="full",
    log_level="DEBUG",
    overwrite=False,
)
paths = prepare_reduction(config, __version__)
logger = setup_logging(config, paths)
print(paths.root)

## 1. Identify observations

The observing log and FITS headers are reconciled first. This stage only decides what data are available and which CCDs should be used.

In [ ]:
reduction_input = observations.identify_observations(config, paths)
display(reduction_input)
print(f"{len(reduction_input)} observations selected for {night}")

## 2. Detector registration

Detector shifts are measured in `detector.py` relative to the reference night. The compact result is written as `detector_shifts_YYMMDD.fits`.


In [ ]:
detector_shifts = detector.measure_detector_shifts(reduction_input, config, paths)
display(detector_shifts)
fits.info(paths.detector_shifts)

## 3. Combine Flats in memory

The individual Flat `DetectorFrame`s are normalized and combined. The full 4112×4096 combined Flat is intentionally **not** saved; it is only an intermediate used to determine geometry and compact 1-D response products.


In [ ]:
combined_flats = flat.combine_flat_frames(reduction_input, config)
for ccd, frame in combined_flats.items():
    print(f"CCD{ccd}: image={frame.image.shape}, finite={np.mean(np.isfinite(frame.image)):.3%}")

## 4. Determine `OrderGeometry`

The reference-night geometry supplies the starting locations. The current Flat and measured detector shifts refine the trace and named cross-dispersion regions. The persistent product has one table row per physical echelle order.


In [ ]:
order_geometry = orders.determine_order_geometry(
    reduction_input, combined_flats, detector_shifts, config, paths
)
order_table = orders.order_geometry_table(order_geometry)
display(order_table[:10])
fits.info(paths.order_geometry)
print(paths.order_geometry)

## 5. Optional compact `FibreGeometry`

For fibre extraction only, the Flat order matrices are fitted at sparse dispersion locations. The saved model contains polynomial coefficients for bundle offset, fibre separation and common Gaussian width, plus one constant offset for each science/sky fibre. The evaluated 4112-row centres are never written to disk.


In [ ]:
flat_order_matrices = flat.extract_flat_order_matrices(combined_flats, order_geometry)
fibre_geometry = {}

if config.extraction_mode == "fibre":
    reference_file = paths.reference_product("fibre_geometry", config.reference_night)
    reference_geometry = fibres.load_fibre_geometry(reference_file) if reference_file.exists() else {}

    if paths.fibre_geometry.exists() and not config.overwrite:
        fibre_geometry = fibres.load_fibre_geometry(paths.fibre_geometry)
    else:
        fibre_geometry = fibres.fit_fibre_geometries(
            flat_order_matrices, config, reference_geometries=reference_geometry
        )
        fibres.save_fibre_geometry(paths.fibre_geometry, fibre_geometry, config)
        fibres.save_fibre_diagnostics(fibre_geometry, flat_order_matrices, config, paths)

    summary = fibres.summarise_fibre_geometry(fibre_geometry)
    display(summary)
    fits.info(paths.fibre_geometry)
    with fits.open(paths.fibre_geometry) as hdul:
        display(Table(hdul["ORDER_MODEL"].data)[:8])
        display(Table(hdul["FIBRE_OFFSETS"].data)[:30])

## 6. Summed and fibre Flat calibrations

For the summed path, the science aperture gives one 4112-pixel Flat spectrum per order; a broad Gaussian-smoothed version is the large-scale illumination/blaze-like shape and their ratio is the small-scale response.

In fibre mode, each extracted fibre is treated independently. `response_fibres` also carries relative fibre-throughput information with respect to the median science-fibre smooth Flat. No 4112×81 multiplicative response map is applied to science/calibration order matrices.


In [ ]:
flat_calibrations = flat.build_flat_calibrations(
    flat_order_matrices, fibre_geometry, config, paths
)

for filename in (
    paths.flat_summed,
    paths.flat_smooth_summed,
    paths.response_summed,
):
    print(filename.name)
    fits.info(filename)

if config.extraction_mode == "fibre":
    for filename in (
        paths.flat_fibres,
        paths.flat_smooth_fibres,
        paths.response_fibres,
    ):
        print(filename.name)
        fits.info(filename)


### Checkpoint: direct summed Flat versus recombined fibres

This is an important independent QA test. The direct summed extraction is retained as the minimally model-dependent reference; wavelength-dependent structure in the fibre/summed ratio can reveal imperfect fibre geometry or deblending.


In [ ]:
if config.extraction_mode == "fibre":
    name = next(name for name in flat_calibrations if name.startswith("ccd_2_"))
    product = flat_calibrations[name]
    geometry = fibre_geometry[name]
    science_components = {str(f) for f in constants.SCIENCE_FIBRES}

    science_idx = np.array([i for i, component in enumerate(geometry.components) if str(component) in science_components], dtype=int)
    recombined = np.nansum(product.fibre_flat[:, science_idx], axis=1)
    scale = np.nanmedian(product.summed_flat / recombined)

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(product.summed_flat, label="direct summed Flat")
    ax.plot(recombined * scale, label="recombined science fibres")
    ax.set(title=name, xlabel="Dispersion pixel", ylabel="Flat counts")
    ax.set_ylim(-1,3)
    ax.legend()
    plt.show()


## 7. Extract calibration spectra in detector coordinates

The same fixed `OrderGeometry` is used for SimTh, SimLC and summed FibTh extraction. Fibre mode additionally extracts the 19 science-fibre FibTh spectra using the fixed `FibreGeometry`. These spectra are deliberately left in detector-pixel coordinates for the next wavelength-calibration stage.


In [ ]:
calibration_exposures = extraction.extract_calibration_exposures(
    reduction_input, order_geometry, fibre_geometry, config
)
calibration_files = extraction.save_calibration_exposures(
    calibration_exposures, paths.calibrations, config.night, overwrite=True
)
for filename in calibration_files:
    print(filename.relative_to(paths.root))


## 8. Wavelength calibration

 Peak measurement and reference-line identification are source-specific (`thorium.py` and `simlc.py`) with `wavelength.py` using the identified line tables to fit the wavelength models.

### 8.1 Reference inputs and bootstrap wavelength solution

The bootstrap solution is used only to identify which laboratory/comb line corresponds to a measured peak. It is not the final nightly wavelength solution. Once the reference night has been validated, `wavelength_static_001122_ccd*.fits` should become the normal bootstrap for subsequent nights.

In [ ]:
Y_BOUNDS = (0.0, 4111.0)
ORDER_BOUNDS = {
    "1": (138, 167),
    "2": (103, 140),
    "3": (65, 104),
}

# Original Murphy atlas + the editable Veloce-specific selection.
murphy_atlas_file = paths.reference_data / "thar_UVES_MM090311.dat"
veloce_atlas_file = paths.reference_data / "veloce_thorium_reference.fits"

if veloce_atlas_file.exists():
    thorium_atlas = thorium.read_veloce_thorium_atlas(veloce_atlas_file)
    print(f"Using curated Veloce Th atlas: {veloce_atlas_file.name}")
else:
    thorium_atlas = thorium.load_murphy_thorium_atlas(murphy_atlas_file)
    print("WARNING: curated Veloce Th atlas not found; using all Murphy Th lines.")

bootstrap_solution = {}
for ccd in ("1", "2", "3"):
    final_reference = (
        paths.reference_data
        / f"wavelength_static_{config.reference_night}_ccd{ccd}.fits"
    )
    legacy_bootstrap = (
        paths.reference_data
        / f"wavelength_bootstrap_{config.reference_night}_ccd{ccd}.fits"
    )
    filename = legacy_bootstrap if legacy_bootstrap.exists() else final_reference
    if not filename.exists():
        raise FileNotFoundError(f"No bootstrap wavelength solution for CCD{ccd}: {filename}")
    bootstrap_solution[ccd], _ = wavelength.read_wavelength_solution_fits(filename)


def detector_shift_y(ccd):
    return detector.detector_shift(detector_shifts, ccd)[1]


def summed_arrays(exposure):
    # extraction stores (4112, n_orders); calibration fitting uses
    # (n_orders, 4112).
    return exposure.summed.flux.T, exposure.summed.variance.T

### 8.2 Calibration-line measurements

#### 8.2.1 FibTh and SimTh

Both thorium sources use the same line measurement/identification machinery. The returned table contains order, measured dispersion position and uncertainty, reference wavelength, quality flags, and detector-space FWHM.

In [ ]:
SIMLC_REPEAT_FREQUENCY_HZ = 25.00000000e9
SIMLC_OFFSET_FREQUENCY_HZ = 9.56000000000e9
USE_SIMLC_FOR_STATIC_SHAPE = True

WAVELENGTH_Y_DEGREES = range(3, 9)
WAVELENGTH_ORDER_DEGREES = range(3, 9)

In [ ]:
thorium_line_sets = {
    source: {ccd: [] for ccd in ("1", "2", "3")}
    for source in ("FibTh", "SimTh")
}

# th_to_run = ['FibTh','SimTh']
th_to_run = ['FibTh']

for source in th_to_run:
    for ccd in ("1", "2", "3"):
        for exposure_index, exposure in enumerate(calibration_exposures[source][ccd]):
            filename = (
                paths.calibrations
                / f"{source.lower()}_lines_{config.night}_run{int(exposure.run):04d}_ccd{ccd}.fits"
            )

            if filename.exists() and not config.overwrite:
                try:
                    result = calibration.read_calibration_line_fits(filename)
                    print(f"Read existing {source} line fit: {filename.name}")
                except Exception as error:
                    print(f"Could not read {filename.name} ({error}); remeasuring")
                    result = None
            else:
                result = None

            if result is None:
                counts, variance = summed_arrays(exposure)
                result = thorium.measure_thorium_lines(
                    counts,
                    exposure.orders,
                    thorium_atlas,
                    variance=variance,
                    source=source,
                    ccd=ccd,
                    exposure_index=exposure_index,
                    mjd_mid=exposure.mjd_mid,
                    reference_wavelength_function=bootstrap_solution[ccd].wavelength,
                    detector_shift_y=detector_shift_y(ccd),
                    y_bounds=Y_BOUNDS,
                    minimum_reference_intensity=1.5,
                    diagnostics=config.diagnostics,
                    diagnostic_dir=paths.figures / "calibration",
                    log_level=config.log_level,
                )
                calibration.write_calibration_line_fits(result, filename, overwrite=True)

            thorium_line_sets[source][ccd].append(result)
            n_used = np.count_nonzero(result.lines["used_for_wavelength_fit"])
            print(
                f"{source} CCD{ccd} run {exposure.run}: "
                f"{n_used}/{len(result.lines)} identified lines retained"
            )


#### 8.2.2 SimLC and the line-spread function

The initial integrated-Gaussian fit is only a peak-detection/centroid seed. `simlc.py` assigns the exact comb mode, infers the shared order-dependent Moffat/empirical eLSF, and then remeasures every identified comb centroid with the adopted profile. The effective detector-space FWHM is retained for the later resolution profile.

In [ ]:
from scipy.interpolate import BSpline
from scipy.optimize import least_squares
from scipy.sparse import lil_matrix


SIMLC_JOINT_MIN_SNR = 15.0
SIMLC_JOINT_MIN_LINES = 20
SIMLC_JOINT_MAX_LINES = 160
SIMLC_JOINT_FIT_HALF_WIDTH = 3

SIMLC_JOINT_FWHM_BOUNDS = (0.7, 3.0)
SIMLC_JOINT_BETA_BOUNDS = (1.2, 10.0)

SIMLC_BETA_GRID = np.array([1.5, 2.0, 2.5, 3.0, 4.0, 5.0])
SIMLC_BETA_REFERENCE = 3.0


SIMLC_JOINT_KNOT_SPACING = 600
SIMLC_JOINT_SPLINE_DEGREE = 3
SIMLC_JOINT_SMOOTHNESS = 100.0

In [ ]:
def _bspline_knots(
    y_bounds=(0, 4111),
    spacing=SIMLC_JOINT_KNOT_SPACING,
    degree=SIMLC_JOINT_SPLINE_DEGREE,
):
    """Clamped B-spline knot vector."""
    lo, hi = map(float, y_bounds)

    internal = np.arange(
        lo + spacing,
        hi,
        spacing,
        dtype=float,
    )

    return np.r_[
        np.repeat(lo, degree + 1),
        internal,
        np.repeat(hi, degree + 1),
    ]


def _bspline_basis(y, knots, degree=SIMLC_JOINT_SPLINE_DEGREE):
    """Evaluate all B-spline basis functions at y."""
    y = np.atleast_1d(np.asarray(y, dtype=float))
    n_coeff = len(knots) - degree - 1

    basis = np.empty((len(y), n_coeff))

    for j in range(n_coeff):
        coefficients = np.zeros(n_coeff)
        coefficients[j] = 1.0

        basis[:, j] = BSpline(
            knots,
            coefficients,
            degree,
            extrapolate=True,
        )(y)

    return basis

In [ ]:
def fit_simlc_joint_moffat_order(
    counts,
    variance,
    orders,
    lines,
    order,
    *,
    minimum_snr=SIMLC_JOINT_MIN_SNR,
    minimum_lines=SIMLC_JOINT_MIN_LINES,
    maximum_lines=SIMLC_JOINT_MAX_LINES,
    fit_half_width=SIMLC_JOINT_FIT_HALF_WIDTH,
    fwhm_bounds=SIMLC_JOINT_FWHM_BOUNDS,
    beta_bounds=SIMLC_JOINT_BETA_BOUNDS,
    knot_spacing=SIMLC_JOINT_KNOT_SPACING,
    smoothness=SIMLC_JOINT_SMOOTHNESS,
    maximum_centroid_shift=0.55,
    fixed_beta=None,
):
    """Jointly fit smooth FWHM(y) and one beta to all usable modes in one order."""

    order = int(order)
    orders = np.asarray(orders, dtype=int)
    order_lookup = {int(m): i for i, m in enumerate(orders)}

    if order not in order_lookup:
        return None

    y_column = "y_initial" if "y_initial" in lines.colnames else "y"
    fwhm_column = (
        "fwhm_initial"
        if "fwhm_initial" in lines.colnames
        else "fwhm"
    )
    snr_column = (
        "signal_to_noise_initial"
        if "signal_to_noise_initial" in lines.colnames
        else "signal_to_noise"
    )

    y = np.asarray(lines[y_column], dtype=float)
    fwhm_initial = np.asarray(lines[fwhm_column], dtype=float)
    snr = np.asarray(lines[snr_column], dtype=float)
    line_order = np.asarray(lines["order"], dtype=int)
    modes = np.asarray(lines["comb_mode"], dtype=np.int64)

    good = (
        (line_order == order)
        & (modes >= 0)
        & np.isfinite(y)
        & np.isfinite(fwhm_initial)
        & (fwhm_initial > 0)
        & np.isfinite(snr)
        & (snr >= minimum_snr)
    )

    if "fit_success_initial" in lines.colnames:
        good &= np.asarray(
            lines["fit_success_initial"],
            dtype=bool,
        )

    if "nearest_peak_distance_pixel" in lines.colnames:
        distance = np.asarray(
            lines["nearest_peak_distance_pixel"],
            dtype=float,
        )

        good &= (
            ~np.isfinite(distance)
            | (distance >= SIMLC_MOFFAT_MIN_NEIGHBOUR_DISTANCE)
        )

    indices = np.flatnonzero(good)

    if len(indices) < minimum_lines:
        return None

    # Keep a well-distributed subset if there are very many modes.
    indices = indices[np.argsort(y[indices])]

    if maximum_lines is not None and len(indices) > maximum_lines:
        select = np.linspace(
            0,
            len(indices) - 1,
            maximum_lines,
        ).round().astype(int)

        indices = indices[select]

    # ------------------------------------------------------------------
    # Extract local pixel windows.
    # ------------------------------------------------------------------

    row = order_lookup[order]
    line_data = []

    for index in indices:
        y0 = float(y[index])
        centre_pixel = int(round(y0))

        lo = centre_pixel - fit_half_width
        hi = centre_pixel + fit_half_width + 1

        if lo < 0 or hi > counts.shape[1]:
            continue

        pixels = np.arange(lo, hi, dtype=float)
        signal = np.asarray(
            counts[row, lo:hi],
            dtype=float,
        )

        if variance is None:
            var = np.ones_like(signal)
        else:
            var = np.asarray(
                variance[row, lo:hi],
                dtype=float,
            )

        finite = (
            np.isfinite(signal)
            & np.isfinite(var)
            & (var > 0)
        )

        if np.count_nonzero(finite) < 7:
            continue

        pixels = pixels[finite]
        signal = signal[finite]
        var = var[finite]

        background0 = float(
            np.nanmedian(
                np.r_[signal[:2], signal[-2:]]
            )
        )

        amplitude0 = max(
            float(
                np.sum(
                    np.clip(
                        signal - background0,
                        0,
                        None,
                    )
                )
            ),
            1.0,
        )

        line_data.append(
            dict(
                index=int(index),
                y0=y0,
                fwhm0=float(fwhm_initial[index]),
                snr=float(snr[index]),
                pixels=pixels,
                signal=signal,
                variance=var,
                amplitude0=amplitude0,
                background0=background0,
            )
        )

    if len(line_data) < minimum_lines:
        return None

    y_fit = np.array(
        [item["y0"] for item in line_data]
    )
    fwhm0 = np.array(
        [item["fwhm0"] for item in line_data]
    )

    # ------------------------------------------------------------------
    # Smooth log-FWHM spline.
    # ------------------------------------------------------------------

    knots = _bspline_knots(
        spacing=knot_spacing,
    )

    basis = _bspline_basis(
        y_fit,
        knots,
    )

    n_coeff = basis.shape[1]

    log_min = np.log(fwhm_bounds[0])
    log_max = np.log(fwhm_bounds[1])

    coefficients0 = np.full(
        n_coeff,
        np.log(np.nanmedian(fwhm0)),
    )

    # Get a robust initial spline from the Gaussian widths.
    def initial_residual(coefficients):
        data = (
            basis @ coefficients
            - np.log(fwhm0)
        ) / 0.20

        penalty = (
            np.sqrt(smoothness)
            * np.diff(coefficients, n=2)
        )

        return np.r_[data, penalty]

    initial_fit = least_squares(
        initial_residual,
        coefficients0,
        bounds=(
            np.full(n_coeff, log_min),
            np.full(n_coeff, log_max),
        ),
        loss="soft_l1",
        f_scale=1.0,
    )

    coefficients0 = initial_fit.x

    # ------------------------------------------------------------------
    # Full joint parameter vector:
    #
    # spline coefficients
    # beta
    # centre, amplitude, background for every line
    # ------------------------------------------------------------------

    p0 = list(coefficients0)
    lower = [log_min] * n_coeff
    upper = [log_max] * n_coeff

    fit_beta = fixed_beta is None

    if fit_beta:
        beta_index = len(p0)

        p0.append(3.0)
        lower.append(beta_bounds[0])
        upper.append(beta_bounds[1])
    else:
        beta_index = None

    line_parameter_start = len(p0)

    for item in line_data:
        p0.extend([
            item["y0"],
            item["amplitude0"],
            item["background0"],
        ])

        lower.extend([
            item["y0"] - maximum_centroid_shift,
            0.0,
            -np.inf,
        ])

        upper.extend([
            item["y0"] + maximum_centroid_shift,
            np.inf,
            np.inf,
        ])

    p0 = np.asarray(p0, dtype=float)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)

    def data_residuals(parameters):
        coefficients = parameters[:n_coeff]

        beta = (
            float(parameters[beta_index])
            if fit_beta
            else float(fixed_beta)
        )

        log_fwhm = basis @ coefficients
        fwhm = np.exp(log_fwhm)

        output = []

        for j, item in enumerate(line_data):
            start = line_parameter_start + 3 * j

            centre, amplitude, background = (
                parameters[start:start + 3]
            )

            alpha = alpha_from_fwhm(
                fwhm[j],
                beta,
            )

            profile = simlc.pixel_integrated_moffat(
                item["pixels"] - centre,
                alpha,
                beta,
            )

            model = background + amplitude * profile

            output.append(
                (model - item["signal"])
                / np.sqrt(item["variance"])
            )

        return np.concatenate(output)

    def residuals(parameters):
        data = data_residuals(parameters)

        coefficients = parameters[:n_coeff]

        penalty = (
            np.sqrt(smoothness)
            * np.diff(coefficients, n=2)
        )

        return np.r_[data, penalty]

    # ------------------------------------------------------------------
    # Sparse Jacobian structure.
    # ------------------------------------------------------------------

    n_data = sum(
        len(item["pixels"])
        for item in line_data
    )
    n_penalty = n_coeff - 2
    n_parameters = len(p0)

    sparsity = lil_matrix(
        (n_data + n_penalty, n_parameters),
        dtype=int,
    )

    row0 = 0

    for j, item in enumerate(line_data):
        npix = len(item["pixels"])
        rows = slice(row0, row0 + npix)

        # Only local B-spline coefficients contribute.
        active_coefficients = np.flatnonzero(
            np.abs(basis[j]) > 0
        )

        for coefficient in active_coefficients:
            sparsity[rows, coefficient] = 1

        if fit_beta:
            sparsity[rows, beta_index] = 1

        start = line_parameter_start + 3 * j
        sparsity[rows, start:start + 3] = 1

        row0 += npix

    for j in range(n_penalty):
        row = n_data + j
        sparsity[row, j:j + 3] = 1

    fit = least_squares(
        residuals,
        p0,
        bounds=(lower, upper),
        jac_sparsity=sparsity.tocsr(),
        x_scale="jac",
        loss="soft_l1",
        f_scale=1.0,
        max_nfev=2000,
    )

    coefficients = fit.x[:n_coeff]

    beta = (
        float(fit.x[beta_index])
        if fit_beta
        else float(fixed_beta)
    )

    data_resid = data_residuals(fit.x)

    residual_median = float(np.nanmedian(data_resid))

    residual_mad = float(
        1.4826
        * np.nanmedian(
            np.abs(data_resid - residual_median)
        )
    )

    residual_p95 = float(
        np.nanpercentile(
            np.abs(data_resid),
            95,
        )
    )

    # Equivalent to the soft_l1 loss used by least_squares,
    # but evaluated on data pixels only (no spline penalty).
    robust_pixel_loss = float(
        np.nanmean(
            2.0 * (
                np.sqrt(1.0 + data_resid**2)
                - 1.0
            )
        )
    )

    dof = max(
        1,
        len(data_resid)
        - len(fit.x),
    )

    reduced_chi2 = float(
        np.sum(data_resid**2) / dof
    )

    # ------------------------------------------------------------------
    # Extract line-by-line results.
    # ------------------------------------------------------------------

    line_rows = []
    offset = np.arange(-8, 8.001, 0.01)

    for j, item in enumerate(line_data):
        start = line_parameter_start + 3 * j
        centre, amplitude, background = (
            fit.x[start:start + 3]
        )

        intrinsic_fwhm = float(
            np.exp(basis[j] @ coefficients)
        )

        alpha = alpha_from_fwhm(
            intrinsic_fwhm,
            beta,
        )

        profile = simlc.pixel_integrated_moffat(
            offset,
            alpha,
            beta,
        )

        effective_fwhm = simlc.effective_fwhm(
            offset,
            profile,
        )

        local_profile = simlc.pixel_integrated_moffat(
            item["pixels"] - centre,
            alpha,
            beta,
        )

        local_model = (
            background
            + amplitude * local_profile
        )

        local_residual = (
            local_model - item["signal"]
        ) / np.sqrt(item["variance"])

        line_rows.append(
            dict(
                line_index=item["index"],
                order=order,
                y_initial=item["y0"],
                y_joint=float(centre),
                y_shift_joint=float(
                    centre - item["y0"]
                ),
                signal_to_noise=item["snr"],
                fwhm_gaussian=item["fwhm0"],
                fwhm_intrinsic=intrinsic_fwhm,
                fwhm_effective=float(effective_fwhm),
                alpha=float(alpha),
                beta=beta,
                reduced_chi2=float(
                    np.mean(local_residual**2)
                ),
            )
        )

    model = dict(
        order=order,
        success=bool(fit.success),
        n_lines=len(line_data),
        beta=beta,
        beta_fixed=not fit_beta,
        reduced_chi2=reduced_chi2,
        robust_pixel_loss=robust_pixel_loss,
        residual_mad=residual_mad,
        residual_p95=residual_p95,
        spline_degree=SIMLC_JOINT_SPLINE_DEGREE,
        knots=knots,
        coefficients=coefficients,
    )

    return model, line_rows


def fit_simlc_beta_grid(
    counts,
    variance,
    orders,
    lines,
    *,
    beta_grid=SIMLC_BETA_GRID,
    reference_beta=SIMLC_BETA_REFERENCE,
):
    """Profile the joint FWHM(y) model over fixed Moffat beta values."""

    summary_rows = []
    all_line_rows = []
    solutions = {}

    for order in np.asarray(orders, dtype=int):

        for beta in beta_grid:

            result = fit_simlc_joint_moffat_order(
                counts,
                variance,
                orders,
                lines,
                order,
                fixed_beta=float(beta),
            )

            if result is None:
                continue

            model, line_rows = result

            solutions[(int(order), float(beta))] = {
                int(row["line_index"]): float(row["y_joint"])
                for row in line_rows
            }

            summary_rows.append(
                dict(
                    order=int(order),
                    beta=float(beta),
                    n_lines=model["n_lines"],
                    robust_pixel_loss=model["robust_pixel_loss"],
                    reduced_chi2=model["reduced_chi2"],
                    residual_mad=model["residual_mad"],
                    residual_p95=model["residual_p95"],
                    centroid_rms_vs_reference=np.nan,
                    centroid_p95_vs_reference=np.nan,
                )
            )

            for row in line_rows:
                row = dict(row)
                row["beta_test"] = float(beta)
                all_line_rows.append(row)

    # -------------------------------------------------------------
    # How much do the fitted centroids actually depend on beta?
    # -------------------------------------------------------------

    for row in summary_rows:

        order = row["order"]
        beta = row["beta"]

        current = solutions[(order, beta)]
        reference = solutions.get(
            (order, float(reference_beta))
        )

        if reference is None:
            continue

        common = sorted(
            set(current) & set(reference)
        )

        if not common:
            continue

        delta = np.array([
            current[i] - reference[i]
            for i in common
        ])

        row["centroid_rms_vs_reference"] = float(
            np.sqrt(np.mean(delta**2))
        )

        row["centroid_p95_vs_reference"] = float(
            np.percentile(
                np.abs(delta),
                95,
            )
        )

    return (
        Table(rows=summary_rows),
        Table(rows=all_line_rows),
    )


def plot_simlc_beta_grid(
    summary,
    filename,
):
    """Show profile quality and centroid sensitivity versus fixed beta."""

    with PdfPages(filename) as pdf:

        for order in np.sort(
            np.unique(summary["order"])
        ):

            q = (
                np.asarray(summary["order"], dtype=int)
                == int(order)
            )

            beta = np.asarray(
                summary["beta"][q],
                dtype=float,
            )

            score = np.asarray(
                summary["robust_pixel_loss"][q],
                dtype=float,
            )

            centroid_rms = np.asarray(
                summary["centroid_rms_vs_reference"][q],
                dtype=float,
            )

            centroid_p95 = np.asarray(
                summary["centroid_p95_vs_reference"][q],
                dtype=float,
            )

            sort = np.argsort(beta)

            beta = beta[sort]
            score = score[sort]
            centroid_rms = centroid_rms[sort]
            centroid_p95 = centroid_p95[sort]

            fig, axes = plt.subplots(
                2,
                1,
                figsize=(7, 6),
                sharex=True,
            )

            # -----------------------------------------------------
            # Relative profile quality.
            # -----------------------------------------------------

            best = np.nanmin(score)

            axes[0].plot(
                beta,
                100.0 * (score / best - 1.0),
                "o-",
            )

            axes[0].axhline(
                0,
                lw=1,
                ls="--",
            )

            axes[0].set(
                ylabel="Excess robust loss [%]",
                title=f"SimLC order {order}",
            )

            # -----------------------------------------------------
            # Centroid sensitivity.
            # -----------------------------------------------------

            axes[1].plot(
                beta,
                1000.0 * centroid_rms,
                "o-",
                label="RMS",
            )

            axes[1].plot(
                beta,
                1000.0 * centroid_p95,
                "s--",
                label="P95",
            )

            axes[1].axvline(
                SIMLC_BETA_REFERENCE,
                lw=1,
                ls=":",
            )

            axes[1].set(
                xlabel=r"Fixed Moffat $\beta$",
                ylabel=r"$\Delta y$ vs $\beta=3$ [mpix]",
            )

            axes[1].legend(frameon=False)

            fig.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)
            

def fit_simlc_joint_moffat_orders(
    counts,
    variance,
    orders,
    lines,
):
    """Joint smooth-FWHM/constant-beta fit for every order."""

    models = {}
    rows = []

    for order in np.asarray(orders, dtype=int):
        result = fit_simlc_joint_moffat_order(
            counts,
            variance,
            orders,
            lines,
            order,
        )

        if result is None:
            continue

        model, line_rows = result

        models[int(order)] = model
        rows.extend(line_rows)

    return models, Table(rows=rows)


def evaluate_joint_moffat_model(model, y):
    """Evaluate intrinsic and effective FWHM for one fitted order."""

    y = np.atleast_1d(np.asarray(y, dtype=float))

    basis = _bspline_basis(
        y,
        model["knots"],
        degree=model["spline_degree"],
    )

    intrinsic = np.exp(
        basis @ model["coefficients"]
    )

    beta = model["beta"]

    offset = np.arange(-8, 8.001, 0.01)

    effective = np.empty_like(intrinsic)

    for i, fwhm in enumerate(intrinsic):
        alpha = alpha_from_fwhm(
            fwhm,
            beta,
        )

        profile = simlc.pixel_integrated_moffat(
            offset,
            alpha,
            beta,
        )

        effective[i] = simlc.effective_fwhm(
            offset,
            profile,
        )

    return intrinsic, effective


def plot_simlc_joint_moffat(
    individual,
    joint_lines,
    models,
    filename,
):
    with PdfPages(filename) as pdf:

        for order in sorted(models):
            model = models[order]

            qi = (
                np.asarray(individual["order"], dtype=int)
                == order
            )

            qj = (
                np.asarray(joint_lines["order"], dtype=int)
                == order
            )

            fig, axes = plt.subplots(
                3,
                1,
                figsize=(10, 8),
                sharex=True,
            )

            y_grid = np.linspace(0, 4111, 300)

            _, effective_grid = (
                evaluate_joint_moffat_model(
                    model,
                    y_grid,
                )
            )

            # ------------------------------------------------------
            # FWHM
            # ------------------------------------------------------

            ax = axes[0]

            if np.any(qi):
                ax.scatter(
                    individual["y_initial"][qi],
                    individual["fwhm_effective"][qi],
                    s=8,
                    alpha=0.2,
                    label="individual Moffat",
                )

            if np.any(qj):
                ax.scatter(
                    joint_lines["y_initial"][qj],
                    joint_lines["fwhm_gaussian"][qj],
                    s=8,
                    alpha=0.15,
                    label="initial Gaussian",
                )

            ax.plot(
                y_grid,
                effective_grid,
                lw=2,
                label="joint smooth FWHM(y)",
            )

            ax.set_ylabel("effective FWHM [pix]")
            ax.legend(frameon=False)

            # ------------------------------------------------------
            # beta
            # ------------------------------------------------------

            ax = axes[1]

            if np.any(qi):
                ax.scatter(
                    individual["y_initial"][qi],
                    individual["beta"][qi],
                    s=8,
                    alpha=0.2,
                    label="individual modes",
                )

            ax.axhline(
                model["beta"],
                lw=2,
                label=fr"joint $\beta={model['beta']:.2f}$",
            )

            ax.set_ylabel(r"Moffat $\beta$")
            ax.legend(frameon=False)

            # ------------------------------------------------------
            # centroid correction
            # ------------------------------------------------------

            ax = axes[2]

            if np.any(qj):
                ax.scatter(
                    joint_lines["y_initial"][qj],
                    joint_lines["y_shift_joint"][qj],
                    s=9,
                    alpha=0.5,
                )

            ax.axhline(0, ls="--", lw=1)

            ax.set(
                xlabel="Dispersion pixel y",
                ylabel=r"$y_{\rm joint}-y_{\rm initial}$ [pix]",
                xlim=(0, 4111),
            )

            fig.suptitle(
                f"SimLC order {order}: "
                f"N={model['n_lines']}, "
                fr"$\beta={model['beta']:.2f}$, "
                fr"$\chi^2_\nu={model['reduced_chi2']:.2f}$"
            )

            fig.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)

In [ ]:
### TEST!

from scipy.optimize import least_squares
from matplotlib.backends.backend_pdf import PdfPages


# -------------------------------------------------------------------------
# Diagnostic: fit an independent pixel-integrated Moffat to each good SimLC
# mode. This is deliberately diagnostic only; these fits are not yet used
# for the adopted SimLC centroids or wavelength solution.
# -------------------------------------------------------------------------

SIMLC_MOFFAT_HALF_WIDTH = 3
SIMLC_MOFFAT_MIN_SNR = 15.0
SIMLC_MOFFAT_MIN_NEIGHBOUR_DISTANCE = 7.2

MOFFAT_ALPHA_BOUNDS = (0.1, 4.0)
MOFFAT_BETA_BOUNDS = (0.1, 7.0)

# Overlapping local LSF fits
SIMLC_LSF_Y_NODES = np.arange(300, 3901, 300)
SIMLC_LSF_WINDOW_HALF_WIDTH = 350
SIMLC_LSF_MIN_LINES = 8

def moffat_intrinsic_fwhm(alpha, beta):
    """Continuous Moffat FWHM before detector-pixel integration."""
    return 2.0 * alpha * np.sqrt(2.0**(1.0 / beta) - 1.0)


def alpha_from_fwhm(fwhm, beta):
    """Alpha corresponding to the continuous Moffat FWHM."""
    return fwhm / (2.0 * np.sqrt(2.0**(1.0 / beta) - 1.0))


def fit_single_simlc_moffat(
    pixels,
    signal,
    variance,
    y0,
    fwhm0,
):
    """Fit amplitude, centre, background, alpha and beta to one comb mode."""

    pixels = np.asarray(pixels, dtype=float)
    signal = np.asarray(signal, dtype=float)

    if variance is None:
        variance = np.ones_like(signal)
    else:
        variance = np.asarray(variance, dtype=float)

    good = (
        np.isfinite(pixels)
        & np.isfinite(signal)
        & np.isfinite(variance)
        & (variance > 0)
    )

    pixels = pixels[good]
    signal = signal[good]
    variance = variance[good]

    if len(pixels) < 7:
        return None

    sigma = np.sqrt(variance)

    background0 = float(np.nanmedian(np.r_[signal[:2], signal[-2:]]))
    amplitude0 = max(
        float(np.sum(np.clip(signal - background0, 0.0, None))),
        1.0,
    )

    beta0 = 2.5
    alpha0 = alpha_from_fwhm(fwhm0, beta0)
    alpha0 = np.clip(
        alpha0,
        MOFFAT_ALPHA_BOUNDS[0] + 1e-3,
        MOFFAT_ALPHA_BOUNDS[1] - 1e-3,
    )

    # amplitude, centre, background, alpha, beta
    p0 = np.array([
        amplitude0,
        y0,
        background0,
        alpha0,
        beta0,
    ])

    lower = np.array([
        0.0,
        y0 - 0.55,
        -np.inf,
        MOFFAT_ALPHA_BOUNDS[0],
        MOFFAT_BETA_BOUNDS[0],
    ])

    upper = np.array([
        np.inf,
        y0 + 0.55,
        np.inf,
        MOFFAT_ALPHA_BOUNDS[1],
        MOFFAT_BETA_BOUNDS[1],
    ])

    def residuals(parameters):
        amplitude, centre, background, alpha, beta = parameters

        profile = simlc.pixel_integrated_moffat(
            pixels - centre,
            alpha,
            beta,
        )

        model = background + amplitude * profile
        return (model - signal) / sigma

    fit = least_squares(
        residuals,
        p0,
        bounds=(lower, upper),
        x_scale="jac",
        max_nfev=2000,
    )

    amplitude, centre, background, alpha, beta = fit.x

    dof = max(1, len(pixels) - len(fit.x))
    reduced_chi2 = float(np.sum(fit.fun**2) / dof)

    # Approximate covariance from the local Jacobian.
    uncertainty = np.full(len(fit.x), np.nan)
    try:
        covariance = np.linalg.pinv(fit.jac.T @ fit.jac)
        covariance *= reduced_chi2
        uncertainty = np.sqrt(np.clip(np.diag(covariance), 0.0, None))
    except np.linalg.LinAlgError:
        pass

    # Intrinsic analytic Moffat FWHM.
    fwhm_intrinsic = moffat_intrinsic_fwhm(alpha, beta)

    # FWHM after integration over detector pixels.
    offset = np.arange(-8.0, 8.001, 0.01)
    profile = simlc.pixel_integrated_moffat(offset, alpha, beta)
    fwhm_effective = simlc.effective_fwhm(offset, profile)

    bound_hit = (
        alpha < MOFFAT_ALPHA_BOUNDS[0] + 0.01
        or alpha > MOFFAT_ALPHA_BOUNDS[1] - 0.01
        or beta < MOFFAT_BETA_BOUNDS[0] + 0.05
        or beta > MOFFAT_BETA_BOUNDS[1] - 0.1
    )

    return dict(
        fit_success=bool(fit.success),
        amplitude=float(amplitude),
        y=float(centre),
        y_shift=float(centre - y0),
        background=float(background),
        alpha=float(alpha),
        alpha_uncertainty=float(uncertainty[3]),
        beta=float(beta),
        beta_uncertainty=float(uncertainty[4]),
        fwhm_intrinsic=float(fwhm_intrinsic),
        fwhm_effective=float(fwhm_effective),
        reduced_chi2=reduced_chi2,
        bound_hit=bool(bound_hit),
    )


def fit_simlc_moffat_peaks(counts, variance, orders, lines):
    """Fit one independent Moffat profile to each suitable SimLC mode."""

    counts = np.asarray(counts, dtype=float)
    orders = np.asarray(orders, dtype=int)
    lookup = {int(order): i for i, order in enumerate(orders)}

    rows = []

    for i in range(len(lines)):
        order = int(lines["order"][i])

        if order not in lookup:
            continue

        if int(lines["comb_mode"][i]) < 0:
            continue

        y0 = float(lines["y_initial"][i])
        fwhm0 = float(lines["fwhm_initial"][i])
        snr = float(lines["signal_to_noise_initial"][i])

        if (
            not np.isfinite(y0)
            or not np.isfinite(fwhm0)
            or fwhm0 <= 0
            or not np.isfinite(snr)
            or snr < SIMLC_MOFFAT_MIN_SNR
        ):
            continue

        if (
            "fit_success_initial" in lines.colnames
            and not bool(lines["fit_success_initial"][i])
        ):
            continue

        if "nearest_peak_distance_pixel" in lines.colnames:
            distance = float(lines["nearest_peak_distance_pixel"][i])
            if (
                np.isfinite(distance)
                and distance < SIMLC_MOFFAT_MIN_NEIGHBOUR_DISTANCE
            ):
                continue

        y_centre = int(round(y0))
        lo = y_centre - SIMLC_MOFFAT_HALF_WIDTH
        hi = y_centre + SIMLC_MOFFAT_HALF_WIDTH + 1

        if lo < 0 or hi > counts.shape[1]:
            continue

        row_index = lookup[order]
        pixels = np.arange(lo, hi, dtype=float)
        signal = counts[row_index, lo:hi]

        local_variance = (
            None
            if variance is None
            else np.asarray(variance[row_index, lo:hi], dtype=float)
        )

        fitted = fit_single_simlc_moffat(
            pixels,
            signal,
            local_variance,
            y0,
            fwhm0,
        )

        if fitted is None:
            continue

        fitted.update(
            order=order,
            line_index=i,
            comb_mode=int(lines["comb_mode"][i]),
            y_initial=y0,
            signal_to_noise=snr,
            fwhm_gaussian=fwhm0,
        )

        rows.append(fitted)

    return Table(rows=rows)


def fit_simlc_shared_moffat_window(
    counts,
    variance,
    orders,
    lines,
    indices,
    *,
    fwhm_bounds=(0.7, 3.0),
    beta_bounds=(0.8, 15.0),
    fit_half_width=3,
    maximum_centroid_shift=0.55,
):
    """
    Fit one shared Moffat FWHM and beta to several SimLC modes.

    Each mode retains its own centre, amplitude and background.
    The shared FWHM is the intrinsic continuous-Moffat FWHM;
    fwhm_effective is calculated after detector-pixel integration.
    """

    lookup = {
        int(order): i
        for i, order in enumerate(np.asarray(orders, dtype=int))
    }

    line_data = []

    for i in indices:
        order = int(lines["order"][i])

        if order not in lookup:
            continue

        y0 = float(
            lines["y_initial"][i]
            if "y_initial" in lines.colnames
            else lines["y"][i]
        )

        y_centre = int(round(y0))
        lo = y_centre - fit_half_width
        hi = y_centre + fit_half_width + 1

        if lo < 0 or hi > counts.shape[1]:
            continue

        pixels = np.arange(lo, hi, dtype=float)
        signal = np.asarray(counts[lookup[order], lo:hi], dtype=float)

        if variance is None:
            var = np.ones_like(signal)
        else:
            var = np.asarray(
                variance[lookup[order], lo:hi],
                dtype=float,
            )

        good = (
            np.isfinite(signal)
            & np.isfinite(var)
            & (var > 0)
        )

        if np.count_nonzero(good) < 7:
            continue

        pixels = pixels[good]
        signal = signal[good]
        var = var[good]

        background0 = float(
            np.nanmedian(np.r_[signal[:2], signal[-2:]])
        )
        amplitude0 = max(
            float(np.sum(np.clip(signal - background0, 0.0, None))),
            1.0,
        )

        line_data.append(
            dict(
                index=int(i),
                y0=y0,
                pixels=pixels,
                signal=signal,
                variance=var,
                amplitude0=amplitude0,
                background0=background0,
            )
        )

    if len(line_data) < SIMLC_LSF_MIN_LINES:
        return None

    # Initial shared FWHM from the Gaussian fits.
    fwhm_column = (
        "fwhm_initial"
        if "fwhm_initial" in lines.colnames
        else "fwhm"
    )

    initial_fwhm = np.array([
        float(lines[fwhm_column][item["index"]])
        for item in line_data
    ])

    fwhm0 = float(np.nanmedian(initial_fwhm))
    fwhm0 = np.clip(
        fwhm0,
        fwhm_bounds[0] + 0.01,
        fwhm_bounds[1] - 0.01,
    )

    beta0 = 3.0

    # parameters:
    # [FWHM, beta,
    #  centre_1, amplitude_1, background_1,
    #  centre_2, amplitude_2, background_2, ...]
    p0 = [fwhm0, beta0]
    lower = [fwhm_bounds[0], beta_bounds[0]]
    upper = [fwhm_bounds[1], beta_bounds[1]]

    for item in line_data:
        y0 = item["y0"]

        p0.extend([
            y0,
            item["amplitude0"],
            item["background0"],
        ])

        lower.extend([
            y0 - maximum_centroid_shift,
            0.0,
            -np.inf,
        ])

        upper.extend([
            y0 + maximum_centroid_shift,
            np.inf,
            np.inf,
        ])

    p0 = np.asarray(p0, dtype=float)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)

    def residuals(parameters):
        fwhm, beta = parameters[:2]

        alpha = alpha_from_fwhm(fwhm, beta)

        output = []

        for j, item in enumerate(line_data):
            centre, amplitude, background = (
                parameters[2 + 3*j : 5 + 3*j]
            )

            profile = simlc.pixel_integrated_moffat(
                item["pixels"] - centre,
                alpha,
                beta,
            )

            model = background + amplitude * profile

            output.append(
                (model - item["signal"])
                / np.sqrt(item["variance"])
            )

        return np.concatenate(output)

    fit = least_squares(
        residuals,
        p0,
        bounds=(lower, upper),
        x_scale="jac",
        max_nfev=5000,
    )

    fwhm, beta = fit.x[:2]
    alpha = alpha_from_fwhm(fwhm, beta)

    dof = max(1, len(fit.fun) - len(fit.x))
    reduced_chi2 = float(
        np.sum(fit.fun**2) / dof
    )

    uncertainty = np.full(2, np.nan)

    try:
        covariance = np.linalg.pinv(
            fit.jac.T @ fit.jac
        )
        covariance *= reduced_chi2

        uncertainty = np.sqrt(
            np.clip(np.diag(covariance)[:2], 0.0, None)
        )
    except np.linalg.LinAlgError:
        pass

    offset = np.arange(-8.0, 8.001, 0.01)

    profile = simlc.pixel_integrated_moffat(
        offset,
        alpha,
        beta,
    )

    fwhm_effective = simlc.effective_fwhm(
        offset,
        profile,
    )

    return dict(
        fit_success=bool(fit.success),
        n_lines=len(line_data),

        fwhm_intrinsic=float(fwhm),
        fwhm_intrinsic_uncertainty=float(uncertainty[0]),

        fwhm_effective=float(fwhm_effective),

        alpha=float(alpha),

        beta=float(beta),
        beta_uncertainty=float(uncertainty[1]),

        reduced_chi2=reduced_chi2,

        bound_hit=bool(
            fwhm < fwhm_bounds[0] + 0.01
            or fwhm > fwhm_bounds[1] - 0.01
            or beta < beta_bounds[0] + 0.05
            or beta > beta_bounds[1] - 0.1
        ),
    )


def fit_simlc_shared_moffat_windows(
    counts,
    variance,
    orders,
    lines,
    *,
    y_nodes=SIMLC_LSF_Y_NODES,
    window_half_width=SIMLC_LSF_WINDOW_HALF_WIDTH,
    minimum_lines=SIMLC_LSF_MIN_LINES,
    minimum_snr=SIMLC_MOFFAT_MIN_SNR,
):
    """Fit overlapping local shared FWHM+beta models along each order."""

    rows = []

    y_column = (
        "y_initial"
        if "y_initial" in lines.colnames
        else "y"
    )

    snr_column = (
        "signal_to_noise_initial"
        if "signal_to_noise_initial" in lines.colnames
        else "signal_to_noise"
    )

    y = np.asarray(lines[y_column], dtype=float)
    snr = np.asarray(lines[snr_column], dtype=float)
    order_values = np.asarray(lines["order"], dtype=int)
    modes = np.asarray(lines["comb_mode"], dtype=np.int64)

    for order in np.asarray(orders, dtype=int):

        base = (
            (order_values == order)
            & (modes >= 0)
            & np.isfinite(y)
            & np.isfinite(snr)
            & (snr >= minimum_snr)
        )

        if "fit_success_initial" in lines.colnames:
            base &= np.asarray(
                lines["fit_success_initial"],
                dtype=bool,
            )

        if "nearest_peak_distance_pixel" in lines.colnames:
            distance = np.asarray(
                lines["nearest_peak_distance_pixel"],
                dtype=float,
            )

            base &= (
                ~np.isfinite(distance)
                | (
                    distance
                    >= SIMLC_MOFFAT_MIN_NEIGHBOUR_DISTANCE
                )
            )

        for y_node in y_nodes:

            q = (
                base
                & (
                    np.abs(y - y_node)
                    <= window_half_width
                )
            )

            indices = np.flatnonzero(q)

            if len(indices) < minimum_lines:
                continue

            result = fit_simlc_shared_moffat_window(
                counts,
                variance,
                orders,
                lines,
                indices,
            )

            if result is None:
                continue

            result.update(
                order=int(order),
                y_node=float(y_node),
                y_median=float(np.nanmedian(y[indices])),
                y_min=float(np.nanmin(y[indices])),
                y_max=float(np.nanmax(y[indices])),
            )

            rows.append(result)

    return Table(rows=rows)


def binned_median(x, y, bin_width=300):
    """Simple running diagnostic of broad position-dependent structure."""

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    edges = np.arange(0, 4112 + bin_width, bin_width)
    centre, median = [], []

    for lo, hi in zip(edges[:-1], edges[1:]):
        q = (
            np.isfinite(x)
            & np.isfinite(y)
            & (x >= lo)
            & (x < hi)
        )

        if np.count_nonzero(q) >= 3:
            centre.append(np.nanmedian(x[q]))
            median.append(np.nanmedian(y[q]))

    return np.asarray(centre), np.asarray(median)


def plot_simlc_per_peak_moffat(table, filename):
    """One PDF page per order showing alpha, beta and FWHM versus y."""

    with PdfPages(filename) as pdf:
        for order in np.sort(np.unique(table["order"])):

            q = (
                np.asarray(table["order"], dtype=int) == int(order)
            )

            y = np.asarray(table["y_initial"][q], dtype=float)
            alpha = np.asarray(table["alpha"][q], dtype=float)
            beta = np.asarray(table["beta"][q], dtype=float)
            fwhm = np.asarray(table["fwhm_effective"][q], dtype=float)
            fwhm_gaussian = np.asarray(
                table["fwhm_gaussian"][q],
                dtype=float,
            )
            bound = np.asarray(table["bound_hit"][q], dtype=bool)

            fig, axes = plt.subplots(
                3,
                1,
                figsize=(10, 8),
                sharex=True,
            )

            # ---------------------------------------------------------
            # alpha
            # ---------------------------------------------------------
            ax = axes[0]
            ax.scatter(
                y[~bound],
                alpha[~bound],
                s=12,
                alpha=0.6,
            )
            ax.scatter(
                y[bound],
                alpha[bound],
                s=20,
                marker="x",
                label="parameter bound",
            )

            bx, by = binned_median(y[~bound], alpha[~bound])
            ax.plot(bx, by, "o-", lw=2, ms=4)

            ax.set(
                ylabel=r"Moffat $\alpha$ [pix]",
                title=f"SimLC order {order}",
            )
            ax.legend(frameon=False)

            # ---------------------------------------------------------
            # beta
            # ---------------------------------------------------------
            ax = axes[1]
            ax.scatter(
                y[~bound],
                beta[~bound],
                s=12,
                alpha=0.6,
            )
            ax.scatter(
                y[bound],
                beta[bound],
                s=20,
                marker="x",
            )

            bx, by = binned_median(y[~bound], beta[~bound])
            ax.plot(bx, by, "o-", lw=2, ms=4)

            ax.set_ylabel(r"Moffat $\beta$")

            # ---------------------------------------------------------
            # FWHM
            # ---------------------------------------------------------
            ax = axes[2]

            ax.scatter(
                y,
                fwhm_gaussian,
                s=10,
                alpha=0.3,
                label="initial Gaussian",
            )

            ax.scatter(
                y[~bound],
                fwhm[~bound],
                s=12,
                alpha=0.7,
                label="pixel-integrated Moffat",
            )

            bx, by = binned_median(y[~bound], fwhm[~bound])
            ax.plot(bx, by, "o-", lw=2, ms=4)

            ax.set(
                xlabel="Dispersion pixel y",
                ylabel="FWHM [pix]",
                xlim=(0, 4111),
            )
            ax.legend(frameon=False)

            fig.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)


def plot_simlc_moffat_windows(
    individual,
    shared,
    filename,
):
    """Compare individual and locally shared Moffat fits."""

    with PdfPages(filename) as pdf:

        for order in np.sort(
            np.unique(individual["order"])
        ):

            qi = (
                np.asarray(individual["order"], dtype=int)
                == int(order)
            )

            qs = (
                np.asarray(shared["order"], dtype=int)
                == int(order)
            )

            if not np.any(qi):
                continue

            y_i = np.asarray(
                individual["y_initial"][qi],
                dtype=float,
            )

            good_i = ~np.asarray(
                individual["bound_hit"][qi],
                dtype=bool,
            )

            fig, axes = plt.subplots(
                3,
                1,
                figsize=(10, 8),
                sharex=True,
            )

            # ---------------------------------------------------------
            # beta
            # ---------------------------------------------------------

            ax = axes[0]

            ax.scatter(
                y_i[good_i],
                np.asarray(individual["beta"][qi])[good_i],
                s=8,
                alpha=0.25,
                label="individual modes",
            )

            if np.any(qs):
                y_s = np.asarray(
                    shared["y_median"][qs],
                    dtype=float,
                )

                beta_s = np.asarray(
                    shared["beta"][qs],
                    dtype=float,
                )

                beta_error = np.asarray(
                    shared["beta_uncertainty"][qs],
                    dtype=float,
                )

                ax.errorbar(
                    y_s,
                    beta_s,
                    yerr=beta_error,
                    marker="o",
                    lw=1.5,
                    label="shared window",
                )

            ax.set(
                ylabel=r"Moffat $\beta$",
                title=f"SimLC order {order}",
            )

            ax.legend(frameon=False)

            # ---------------------------------------------------------
            # effective FWHM
            # ---------------------------------------------------------

            ax = axes[1]

            ax.scatter(
                y_i[good_i],
                np.asarray(
                    individual["fwhm_effective"][qi],
                    dtype=float,
                )[good_i],
                s=8,
                alpha=0.25,
                label="individual Moffat",
            )

            if "fwhm_gaussian" in individual.colnames:
                ax.scatter(
                    y_i,
                    np.asarray(
                        individual["fwhm_gaussian"][qi],
                        dtype=float,
                    ),
                    s=7,
                    alpha=0.15,
                    label="initial Gaussian",
                )

            if np.any(qs):
                ax.plot(
                    y_s,
                    np.asarray(
                        shared["fwhm_effective"][qs],
                        dtype=float,
                    ),
                    "o-",
                    lw=1.5,
                    label="shared window",
                )

            ax.set_ylabel("effective FWHM [pix]")
            ax.legend(frameon=False)

            # ---------------------------------------------------------
            # lines / fit quality
            # ---------------------------------------------------------

            ax = axes[2]

            if np.any(qs):
                ax.plot(
                    y_s,
                    np.asarray(
                        shared["n_lines"][qs],
                        dtype=float,
                    ),
                    "o-",
                    label="lines/window",
                )

                ax.set_ylabel("number of modes")

                ax2 = ax.twinx()

                ax2.plot(
                    y_s,
                    np.asarray(
                        shared["reduced_chi2"][qs],
                        dtype=float,
                    ),
                    "s--",
                    alpha=0.7,
                )

                ax2.set_ylabel(
                    r"reduced $\chi^2$"
                )

            ax.set(
                xlabel="Dispersion pixel y",
                xlim=(0, 4111),
            )

            fig.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)

In [ ]:
simlc_line_sets = {ccd: [] for ccd in ("2", "3")}

for ccd in ("2", "3"):
    for exposure_index, exposure in enumerate(calibration_exposures["SimLC"][ccd]):
        line_filename = (
            paths.calibrations
            / f"simlc_lines_{config.night}_run{int(exposure.run):04d}_ccd{ccd}.fits"
        )
        lsf_filename = (
            paths.calibrations
            / f"simlc_lsf_{config.night}_run{int(exposure.run):04d}_ccd{ccd}.fits"
        )

        if line_filename.exists() and not config.overwrite:
            try:
                result = calibration.read_calibration_line_fits(line_filename)
                if lsf_filename.exists():
                    result.lsf, _ = simlc.read_simlc_lsf_fits(lsf_filename)
                print(f"Read existing SimLC line fit: {line_filename.name}")
            except Exception as error:
                print(f"Could not read {line_filename.name} ({error}); remeasuring")
                result = None
        else:
            result = None

        counts, variance = summed_arrays(exposure)

        if result is None:
            result = simlc.measure_simlc_lines(
                counts,
                exposure.orders,
                variance=variance,
                ccd=ccd,
                exposure_index=exposure_index,
                mjd_mid=exposure.mjd_mid,
                reference_wavelength_function=bootstrap_solution[ccd].wavelength,
                detector_shift_y=detector_shift_y(ccd),
                y_bounds=Y_BOUNDS,
                repetition_rate_hz=SIMLC_REPEAT_FREQUENCY_HZ,
                offset_frequency_hz=SIMLC_OFFSET_FREQUENCY_HZ,
                diagnostics=config.diagnostics,
                diagnostic_dir=paths.figures / "calibration",
                log_level=config.log_level,
            )

            calibration.write_calibration_line_fits(result, line_filename, overwrite=True)

            if result.lsf is not None:
                simlc.write_simlc_lsf_fits(
                    result.lsf,
                    lsf_filename,
                    ccd=ccd,
                    mjd_mid=exposure.mjd_mid,
                    source_peak_file=line_filename,
                    overwrite=True,
                )

        # ---------------------------------------------------------------------
        # Diagnostic Moffat experiments.
        # ---------------------------------------------------------------------

        # Independent alpha/beta fit to each suitable comb mode.
        moffat_peaks = fit_simlc_moffat_peaks(
            counts,
            variance,
            exposure.orders,
            result.lines,
        )

        moffat_filename = (
            paths.calibrations
            / f"simlc_moffat_peaks_{config.night}_"
            f"run{int(exposure.run):04d}_ccd{ccd}.fits"
        )

        moffat_peaks.write(
            moffat_filename,
            overwrite=True,
        )

        beta_summary, beta_lines = fit_simlc_beta_grid(
            counts,
            variance,
            exposure.orders,
            result.lines,
        )

        beta_summary.write(
            paths.calibrations
            / f"simlc_beta_grid_{config.night}_"
            f"run{int(exposure.run):04d}_ccd{ccd}.fits",
            overwrite=True,
        )

        beta_lines.write(
            paths.calibrations
            / f"simlc_beta_grid_lines_{config.night}_"
            f"run{int(exposure.run):04d}_ccd{ccd}.fits",
            overwrite=True,
        )

        plot_simlc_beta_grid(
            beta_summary,
            paths.figures
            / f"simlc_beta_grid_{config.night}_"
            f"run{int(exposure.run):04d}_ccd{ccd}.pdf",
        )

        joint_models, joint_lines = fit_simlc_joint_moffat_orders(
            counts,
            variance,
            exposure.orders,
            result.lines,
        )

        joint_filename = (
            paths.calibrations
            / f"simlc_moffat_joint_{config.night}_"
            f"run{int(exposure.run):04d}_ccd{ccd}.fits"
        )

        joint_lines.write(
            joint_filename,
            overwrite=True,
        )

        plot_simlc_joint_moffat(
            moffat_peaks,
            joint_lines,
            joint_models,
            paths.figures
            / f"simlc_moffat_joint_{config.night}_"
            f"run{int(exposure.run):04d}_ccd{ccd}.pdf",
        )

        print(
            f"SimLC CCD{ccd} run {exposure.run}: "
            f"{len(joint_models)} orders jointly fitted; "
            f"{len(joint_lines)} modes"
        )
        simlc_line_sets[ccd].append(result)

        # Always make the per-order SimLC peak PDF when diagnostics are enabled,
        # even when the fitted line product was read from cache.  This reruns only
        # candidate detection (background/SNR/candidate locations), not the line
        # fits, and overlays those candidates on the already-saved fitted table.
        if config.diagnostics != "none":
            counts, _ = summed_arrays(exposure)
            diagnostics.save_calibration_order_diagnostics_from_counts(
                counts,
                exposure.orders,
                result.lines,
                config=calibration.CalibrationPeakConfig(),
                calibration_type="SimLC",
                ccd=ccd,
                exposure_index=exposure_index,
                diagnostic_dir=paths.figures / "calibration",
            )

        if result.lsf is not None and config.diagnostics != "none":
            simlc.plot_simlc_lsf_qa(
                result.lsf,
                result.lines,
                filename=(
                    paths.figures
                    / f"simlc_lsf_{config.night}_run{int(exposure.run):04d}_ccd{ccd}.png"
                ),
            )

        n_used = np.count_nonzero(result.lines["used_for_wavelength_fit"])
        print(
            f"SimLC CCD{ccd} run {exposure.run}: "
            f"{n_used}/{len(result.lines)} modes retained"
        )


### 8.3 Static summed wavelength solution

For the first static solution, choose the FibTh exposure closest to the median calibration time and fit the global $m\lambda(y,m)$ surface. The Legendre degrees are selected by spatially blocked cross-validation rather than from the training residuals.

This is intentionally the **FibTh-only baseline**. The next implementation step is to transform SimLC positions onto the summed-science coordinate and replace the FibTh constraints order-by-order where reliable SimLC coverage exists on CCDs 2 and 3.

In [ ]:
all_calibration_mjds = [
    exposure.mjd_mid
    for source in calibration_exposures.values()
    for ccd_exposures in source.values()
    for exposure in ccd_exposures
    if np.isfinite(exposure.mjd_mid)
]
t0 = float(np.median(all_calibration_mjds))
print(f"Wavelength reference epoch: MJD {t0:.8f}")

static_wavelength = {}
static_fitted_lines = {}
static_validation = {}
static_reference_exposure = {}
static_reference_index = {}

for ccd in ("1", "2", "3"):
    candidates = calibration_exposures["FibTh"][ccd]
    if not candidates:
        print(f"CCD{ccd}: no FibTh exposure available")
        continue

    exposure_index = int(
        np.argmin([abs(exposure.mjd_mid - t0) for exposure in candidates])
    )
    exposure = candidates[exposure_index]
    fibth_lines = thorium_line_sets["FibTh"][ccd][exposure_index].lines
    static_reference_exposure[ccd] = exposure
    static_reference_index[ccd] = exposure_index

    fit_input = fibth_lines
    calibration_type = "FibTh"

    # A preliminary FibTh solution is needed only if a SimLC exposure is to be
    # transferred into the FibTh coordinate system.
    if USE_SIMLC_FOR_STATIC_SHAPE and ccd in simlc_line_sets and simlc_line_sets[ccd]:
        preliminary, _, _, _ = wavelength.fit_validated_wavelength_from_peak_table(
            fibth_lines,
            y_bounds=Y_BOUNDS,
            order_bounds=ORDER_BOUNDS[ccd],
            y_degrees=WAVELENGTH_Y_DEGREES,
            order_degrees=WAVELENGTH_ORDER_DEGREES,
            n_folds=5,
            y_blocks=10,
        )
        lc_index = int(
            np.argmin([
                abs(calibration_exposures["SimLC"][ccd][i].mjd_mid - exposure.mjd_mid)
                for i in range(len(simlc_line_sets[ccd]))
            ])
        )
        fit_input, transfer, transferred_lc = wavelength.build_hybrid_static_peak_table(
            fibth_lines,
            simlc_line_sets[ccd][lc_index],
            preliminary.solution,
            y_bounds=Y_BOUNDS,
            order_bounds=ORDER_BOUNDS[ccd],
            transfer_y_degree=2,
            transfer_order_degree=1,
        )
        calibration_type = "FibTh+SimLC"
        transferred_lc.write(
            paths.calibrations / f"simlc_transferred_static_{config.night}_ccd{ccd}.fits",
            overwrite=True,
        )

    fit, fitted_lines, validation, chosen = (
        wavelength.fit_validated_wavelength_from_peak_table(
            fit_input,
            y_bounds=Y_BOUNDS,
            order_bounds=ORDER_BOUNDS[ccd],
            y_degrees=WAVELENGTH_Y_DEGREES,
            order_degrees=WAVELENGTH_ORDER_DEGREES,
            n_folds=5,
            y_blocks=10,
        )
    )

    static_wavelength[ccd] = fit.solution
    static_fitted_lines[ccd] = fitted_lines
    static_validation[ccd] = validation

    validation.write(
        paths.calibrations / f"wavelength_surface_cv_{config.night}_ccd{ccd}.fits",
        overwrite=True,
    )
    diagnostics.print_wavelength_degree_summary(validation, chosen, ccd=ccd)
    if config.diagnostics != "none":
        diagnostics.save_wavelength_degree_diagnostics(
            validation,
            chosen,
            paths.figures / f"wavelength_surface_cv_{config.night}_ccd{ccd}.png",
            calibration_type=calibration_type,
            ccd=ccd,
            diagnostics=config.diagnostics,
        )
    wavelength.write_wavelength_fit_fits(
        fit,
        fitted_lines,
        paths.calibrations / f"wavelength_static_{config.night}_ccd{ccd}.fits",
        calibration_type=calibration_type,
        ccd=ccd,
        mjd_mid=exposure.mjd_mid,
        source_peak_file=(
            f"fibth_lines_{config.night}_run{int(exposure.run):04d}_ccd{ccd}.fits"
        ),
        overwrite=True,
    )

    diagnostics.print_wavelength_fit_summary(
        fit, fitted_lines, calibration_type=calibration_type, ccd=ccd,
    )
    if config.diagnostics != "none":
        diagnostics.save_wavelength_diagnostics(
            fit,
            fitted_lines,
            paths.figures / f"wavelength_static_{config.night}_ccd{ccd}.png",
            calibration_type=calibration_type,
            ccd=ccd,
            diagnostics=config.diagnostics,
        )

    print(
        f"CCD{ccd}: {calibration_type}; degrees "
        f"({int(chosen['y_degree'])}, {int(chosen['order_degree'])}); "
        f"CV RMS={float(chosen['validation_rms_pixel']):.4f} pix"
    )

### 8.4 Fibre-dependent wavelength measurements

The reference FibTh exposure can already be measured fibre-by-fibre with the same thorium code. These line tables are the input to the forthcoming `fit_fibre_corrections()` model; no separate peak-fitting code belongs in `wavelength.py`.

In [ ]:
fibth_fibre_line_sets = {ccd: {} for ccd in ("1", "2", "3")}
fibre_shift_model = {ccd: None for ccd in ("1", "2", "3")}
fibre_shift_qa = {}

if config.extraction_mode == "fibre":
    for ccd, exposure in static_reference_exposure.items():
        if exposure.fibre_flux is None:
            continue

        for fibre_index, fibre_name in enumerate(extraction.SCIENCE_FIBRES):
            counts = exposure.fibre_flux[:, :, fibre_index].T
            variance = exposure.fibre_variance[:, :, fibre_index].T
            result = thorium.measure_thorium_lines(
                counts,
                exposure.orders,
                thorium_atlas,
                variance=variance,
                source="FibTh",
                ccd=ccd,
                exposure_index=static_reference_index[ccd],
                mjd_mid=exposure.mjd_mid,
                fibre=int(fibre_name),
                reference_wavelength_function=static_wavelength[ccd].wavelength,
                detector_shift_y=0.0,
                y_bounds=Y_BOUNDS,
                minimum_reference_intensity=1.5,
                diagnostics="none",  # avoid hundreds of per-fibre QA pages
                log_level=config.log_level,
            )
            fibth_fibre_line_sets[ccd][int(fibre_name)] = result
            calibration.write_calibration_line_fits(
                result,
                paths.calibrations
                / f"fibth_fibre{int(fibre_name):+03d}_{config.night}_ccd{ccd}.fits",
                overwrite=True,
            )

        if fibth_fibre_line_sets[ccd]:
            model, qa, fitted = wavelength.fit_fibre_corrections(
                fibth_fibre_line_sets[ccd],
                static_wavelength[ccd],
                y_bounds=Y_BOUNDS,
                order_bounds=ORDER_BOUNDS[ccd],
                y_degree=2,
                order_degree=1,
            )
            fibre_shift_model[ccd] = model
            fibre_shift_qa[ccd] = qa
            qa.write(
                paths.calibrations / f"wavelength_fibre_qa_{config.night}_ccd{ccd}.fits",
                overwrite=True,
            )
            print(
                f"CCD{ccd}: fibre correction from {len(model.surfaces)} science fibres; "
                f"median RMS={np.nanmedian(qa['rms_pixel']):.4f} pix"
            )


### 8.5 Temporal wavelength corrections

All SimTh and SimLC exposures have already been reduced to homogeneous line tables above. The forthcoming temporal model will compare each source against its own reference epoch so that the fixed SimTh/SimLC fibre offset is not confused with temporal drift.

In [ ]:
time_shift_model = {ccd: None for ccd in ("1", "2", "3")}
time_shift_qa = {}
time_shift_series = {}
wavelength_model = {}

for ccd in static_wavelength:
    lc_sets = simlc_line_sets.get(ccd, [])
    th_sets = thorium_line_sets["SimTh"].get(ccd, [])

    try:
        time_model, qa, all_series = wavelength.fit_time_corrections(
            static_solution=static_wavelength[ccd],
            simlc_line_sets=lc_sets,
            simth_line_sets=th_sets,
            reference_mjd=t0,
            y_bounds=Y_BOUNDS,
            order_bounds=ORDER_BOUNDS[ccd],
            y_degree=2,
            order_degree=1,
            preferred_source="SimLC",
        )
        time_shift_model[ccd] = time_model
        time_shift_qa[ccd] = qa
        time_shift_series[ccd] = all_series
        qa.write(
            paths.calibrations / f"wavelength_time_qa_{config.night}_ccd{ccd}.fits",
            overwrite=True,
        )
        print(
            f"CCD{ccd}: temporal source={time_model.source}, "
            f"N={len(time_model.mjd)}, reference MJD={time_model.reference_mjd:.8f}"
        )
    except RuntimeError as error:
        print(f"CCD{ccd}: no temporal correction ({error})")

    wavelength_model[ccd] = wavelength.WavelengthCalibration(
        static=static_wavelength[ccd],
        fibre=fibre_shift_model.get(ccd),
        time=time_shift_model.get(ccd),
    )
    wavelength.write_wavelength_model_fits(
        wavelength_model[ccd],
        paths.calibrations / f"wavelength_model_{config.night}_ccd{ccd}.fits",
        ccd=ccd,
        reference_mjd=t0,
        overwrite=True,
    )


### 8.6 Compact QA / usage check

In [ ]:
for ccd, model in wavelength_model.items():
    m0 = int(np.round(np.mean(ORDER_BOUNDS[ccd])))
    y0 = 0.5 * sum(Y_BOUNDS)
    message = [f"CCD{ccd}: lambda({y0:.1f}, m={m0})={model.static.wavelength(y0, m0):.6f} nm"]
    if model.time is not None:
        drift = [float(model.time.shift(y0, m0, t)) for t in model.time.mjd]
        message.append(
            f"{model.time.source} drift range=[{np.min(drift):+.4f}, {np.max(drift):+.4f}] pix"
        )
    print("; ".join(message))

# Example for a science/fibre spectrum:
# y = np.arange(4112, dtype=float)
# wave_nm = wavelength_model[ccd].wavelength(
#     y, order, fibre=int(fibre_name), mjd=science_mjd_mid
# )

### 9 Resolution profile

The peak tables persist `fwhm_pixel` and `fwhm_uncertainty_pixel`. Once the final wavelength model is fixed, the local wavelength FWHM and resolving power can be derived from

$$
\Delta\lambda_{\rm FWHM}
=
\left|\frac{d\lambda}{dy}\right|
{\rm FWHM}_{\rm pix},
\qquad
\mathcal{R}
=
\frac{\lambda}{\Delta\lambda_{\rm FWHM}}.
$$

The smooth $\mathcal{R}(y,m)$ model should therefore be fitted after the wavelength solution rather than stored as a primary line-fit quantity.

In [ ]:
# resolution_measurements = {}

# for ccd, lines in static_fitted_lines.items():
#     used = np.asarray(lines["used_for_wavelength_fit"], bool)
#     y = np.asarray(lines["y"], float)
#     order = np.asarray(lines["order"], int)
#     fwhm_pixel = np.asarray(lines["fwhm_pixel"], float)

#     wavelength_nm = static_wavelength[ccd].wavelength(y, order)
#     dispersion_nm_per_pixel = np.abs(
#         static_wavelength[ccd].dispersion(y, order)
#     )
#     delta_lambda_nm = dispersion_nm_per_pixel * fwhm_pixel
#     resolving_power = np.divide(
#         wavelength_nm,
#         delta_lambda_nm,
#         out=np.full_like(wavelength_nm, np.nan),
#         where=np.isfinite(delta_lambda_nm) & (delta_lambda_nm > 0),
#     )

#     resolution_measurements[ccd] = Table(
#         {
#             "order": order[used],
#             "y": y[used],
#             "wavelength_nm": wavelength_nm[used],
#             "fwhm_pixel": fwhm_pixel[used],
#             "resolving_power": resolving_power[used],
#         }
#     )

#     display(resolution_measurements[ccd][:5])

# # NEXT: fit a smooth resolution profile in (y, m).

## 9. Science extraction


In [ ]:
# This needs to be updated later, once we have all the calibration products, to extract and calibrate thescience exposures

# science_exposures = science.extract_science_exposures(
#     reduction_input, nightly_tramlines, flat_products,
#     wavelength_model, config, paths
# )
# print(f"{len(science_exposures)} science CCD exposures")

# if science_exposures:
#     exposure = science_exposures[0]
#     order = exposure.orders[len(exposure.orders) // 2]
#     plt.figure(figsize=(10, 3))
#     plt.plot(order.barycentric_wavelength_nm, order.flux)
#     plt.xlabel("Barycentric wavelength / nm")
#     plt.ylabel("Flux")
#     plt.title(f"{exposure.object_name} — CCD{exposure.ccd}, order {order.order}")
#     plt.show()


## One-call equivalent


In [ ]:
# from velocereduction import pipeline
# state = pipeline.reduce_night(config, version=__version__)
